## 含并行连结的网络
 GoogLeNet吸收了NiN中串联网络的思想，并在此基础上做了改进。 这篇论文的一个重点是解决了什么样大小的卷积核最合适的问题。 毕竟，以前流行的网络使用小到1x1,大到11x11的卷积核，本文的一个观点是，有时使用不同大小的卷积核组合是有利的。
### Inception块
在GoogLeNet中，基本的卷积块被称为Inception块（Inception block）。在Inception块中，通常调整的超参数是每层输出通道数。

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

class Inception(nn.Module):
    # c1-c4是每条路径的输出通道数
    def __init__(self, in_channels,c1,c2,c3,c4, **kwargs) -> None:
        super(Inception,self).__init__( **kwargs)
        # 线路1，单1x1卷积层
        self.p1_1 = nn.Conv2d(in_channels,c1,kernel_size=1)
        # 线路2，1x1卷积层后接3x3卷积层
        self.p2_1 = nn.Conv2d(in_channels,c2[0],kernel_size=3,padding=1)
        self.p2_2 = nn.Conv2d(c2[0],c2[1],kernel_size=3,padding=1)
        # 线路3,1x1卷积层后接5x5卷积层
        self.P3_1 = nn.Conv2d(in_channels,c3[0],kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0],c3[1],kernel_size=5,padding=2)
        # 线路4，3x3最大汇聚层后接1x1卷积层
        self.p4_1 = nn.MaxPool2d(kernel_size=3,stride=1,padding=1)
        self.P4_2 = nn.Conv2d(in_channels,c4,kernel_size=1)
    
    def forward(self,x):
        p1 = F.relu(self.p1_1(x))
        p2 = F.relu(self.p2_2(F.relu(self.p2_1(x))))
        p3 = F.relu(self.p3_2(F.relu(self.P3_1(x))))
        p4 = F.relu(self.P4_2(self.p4_1(x)))
        # 在通道维度上连结输出
        return torch.cat((p1,p2,p3,p4),dim=1)

那么为什么GoogLeNet这个网络如此有效呢？ 首先我们考虑一下滤波器（filter）的组合，它们可以用各种滤波器尺寸探索图像，这意味着不同大小的滤波器可以有效地识别不同范围的图像细节。 同时，我们可以为不同的滤波器分配不同数量的参数。
### GoogLeNet模型
